# Portable Model Experiment

이 노트북은 `CurrentEquipmentDays`와 여러 도메인 특화 컬럼을 제거한 뒤, 범용성이 더 높은 feature set으로 다시 학습한 실험이다.

제거 기준:
- `CurrentEquipmentDays`
- `ServiceArea`
- `PrizmCode`
- `Handsets`, `HandsetModels`, `HandsetRefurbished`, `HandsetWebCapable`, `HandsetPrice`
- `NewCellphoneUser`, `NotNewCellphoneUser`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from xgboost import XGBClassifier

ROOT = Path('/home/ms990728/소정해')
DATA_PATH = ROOT / 'data/raw/cell2celltrain.csv'
RANDOM_STATE = 42

BEST_PARAMS = {
    'colsample_bytree': 0.8334271527847554,
    'gamma': 0.11429345433755263,
    'learning_rate': 0.06857996256539675,
    'max_depth': 3,
    'min_child_weight': 2,
    'n_estimators': 493,
    'reg_alpha': 0.2497327922401265,
    'reg_lambda': 0.9246782213565523,
    'subsample': 0.7954562418017752,
}

REMOVE = {
    'CustomerID', 'Churn',
    'CurrentEquipmentDays', 'ServiceArea', 'PrizmCode',
    'Handsets', 'HandsetModels', 'HandsetRefurbished', 'HandsetWebCapable', 'HandsetPrice',
    'NewCellphoneUser', 'NotNewCellphoneUser',
}

@dataclass
class ResultRow:
    model: str
    train_seconds: float
    roc_auc: float
    accuracy: float
    precision: float
    recall: float
    f1: float


def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, na_values=['NA', ''], keep_default_na=True)
    df['HandsetPrice'] = pd.to_numeric(df['HandsetPrice'].replace('Unknown', pd.NA), errors='coerce')
    df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1}).astype('Int64')
    return df


def build_feature_sets(df: pd.DataFrame):
    feature_df = df.drop(columns=[c for c in REMOVE if c in df.columns])
    target = df['Churn'].astype(int)
    numeric_cols = [col for col in feature_df.columns if pd.api.types.is_numeric_dtype(feature_df[col])]
    categorical_cols = [col for col in feature_df.columns if col not in numeric_cols]
    return feature_df, target, numeric_cols, categorical_cols


def build_logistic_pipeline(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler()),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ])
    model = LogisticRegression(max_iter=10000, solver='saga', class_weight='balanced', random_state=RANDOM_STATE)
    return Pipeline([('preprocess', preprocessor), ('model', model)])


def build_tree_pipeline(numeric_cols, categorical_cols, estimator):
    numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ])
    return Pipeline([('preprocess', preprocessor), ('model', estimator)])


def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test):
    start = perf_counter()
    pipeline.fit(X_train, y_train)
    train_seconds = perf_counter() - start
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    return ResultRow(
        model=name,
        train_seconds=train_seconds,
        roc_auc=roc_auc_score(y_test, y_prob),
        accuracy=accuracy_score(y_test, y_pred),
        precision=precision_score(y_test, y_pred, zero_division=0),
        recall=recall_score(y_test, y_pred, zero_division=0),
        f1=f1_score(y_test, y_pred, zero_division=0),
    )


def best_threshold_from_oof(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = []
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        scores.append(f1_score(y_true, pred, zero_division=0))
    best_idx = int(np.argmax(scores))
    return float(thresholds[best_idx]), float(scores[best_idx])


def fmt(results):
    header = '| Model | Train sec | ROC-AUC | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|
'
    rows = [
        f'| {r.model} | {r.train_seconds:.1f} | {r.roc_auc:.4f} | {r.accuracy:.4f} | {r.precision:.4f} | {r.recall:.4f} | {r.f1:.4f} |'
        for r in results
    ]
    return header + '
'.join(rows) + '
'


df = load_data(DATA_PATH)
feature_df, target, numeric_cols, categorical_cols = build_feature_sets(df)
X_train, X_test, y_train, y_test = train_test_split(
    feature_df,
    target,
    test_size=0.2,
    stratify=target,
    random_state=RANDOM_STATE,
)
pos = int(y_train.sum())
neg = int((y_train == 0).sum())
scale_pos_weight = neg / max(pos, 1)
print('Portable set rows:', len(df))
print('Portable features used:', len(feature_df.columns))
print('Numeric features:', len(numeric_cols))
print('Categorical features:', len(categorical_cols))
print('Positive class weight:', round(scale_pos_weight, 4))


In [ ]:
logistic = build_logistic_pipeline(numeric_cols, categorical_cols)
rf = build_tree_pipeline(
    numeric_cols,
    categorical_cols,
    RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
)
xgb = build_tree_pipeline(
    numeric_cols,
    categorical_cols,
    XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
)

baseline_results = [
    evaluate_model('Logistic Regression', logistic, X_train, y_train, X_test, y_test),
    evaluate_model('Random Forest', rf, X_train, y_train, X_test, y_test),
    evaluate_model('XGBoost', xgb, X_train, y_train, X_test, y_test),
]

print('Portable baseline results')
print(fmt(baseline_results))


In [ ]:
best_xgb = build_tree_pipeline(
    numeric_cols,
    categorical_cols,
    XGBClassifier(
        **BEST_PARAMS,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_prob = cross_val_predict(best_xgb, X_train, y_train, cv=cv, method='predict_proba', n_jobs=1)[:, 1]
best_threshold, best_oof_f1 = best_threshold_from_oof(y_train.to_numpy(), oof_prob)
oof_auc = roc_auc_score(y_train, oof_prob)
best_xgb.fit(X_train, y_train)
test_prob = best_xgb.predict_proba(X_test)[:, 1]
default_pred = (test_prob >= 0.5).astype(int)
tuned_pred = (test_prob >= best_threshold).astype(int)

print('Portable tuned XGBoost')
print('OOF ROC-AUC:', oof_auc)
print('OOF F1:', best_oof_f1)
print('Best threshold:', best_threshold)
print('Default metrics:', {
    'roc_auc': roc_auc_score(y_test, test_prob),
    'accuracy': accuracy_score(y_test, default_pred),
    'precision': precision_score(y_test, default_pred, zero_division=0),
    'recall': recall_score(y_test, default_pred, zero_division=0),
    'f1': f1_score(y_test, default_pred, zero_division=0),
})
print('Tuned metrics:', {
    'roc_auc': roc_auc_score(y_test, test_prob),
    'accuracy': accuracy_score(y_test, tuned_pred),
    'precision': precision_score(y_test, tuned_pred, zero_division=0),
    'recall': recall_score(y_test, tuned_pred, zero_division=0),
    'f1': f1_score(y_test, tuned_pred, zero_division=0),
})


## 결과 요약

- Portable set features used: `46`
- Baseline XGBoost ROC-AUC: `0.6688`
- Baseline XGBoost F1: `0.4785`
- Tuned XGBoost ROC-AUC: `0.6722`
- Tuned XGBoost F1: `0.4946`
- Best threshold: `0.445`

해석:
- `CurrentEquipmentDays` 하나만 뺀 것보다 포터블 세트를 더 넓게 줄였을 때 성능은 추가로 조금 내려간다.
- 다만 portable 방향에서는 이 정도 하락을 감수하고 외부 데이터 일반화 가능성을 확보하는 게 맞다.
